In [19]:

import os
import pandas as pd
import numpy as np
from torchvision import transforms
from PIL import Image, UnidentifiedImageError
from torch.utils.data import Dataset, DataLoader
import torch
import torch.nn as nn
import torchvision.models as models
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import roc_auc_score

import torchvision  # Add this import statement
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Define paths
checkpoint_dir = "/mnt/Internal/MedImage/CheXpert Dataset/Lab_Rotation_2/resnet50_Real_vs_real_training_for_14_disease/"
best_model_path = os.path.join(checkpoint_dir, 'model_epoch_3.pth')

disease_names = ['No Finding', 'Enlarged Cardiomediastinum', 'Cardiomegaly', 'Lung Opacity', 'Lung Lesion', 'Edema',
                 'Consolidation', 'Pneumonia', 'Atelectasis', 'Pneumothorax', 'Pleural Effusion', 'Pleural Other',
                 'Fracture', 'Support Devices']


from torchvision import models
import torch.nn as nn
################For Denset use this################
# model = torchvision.models.densenet121(pretrained=False)  
# num_ftrs = model.classifier.in_features  
# model.classifier = nn.Linear(num_ftrs, len(disease_names))  
# model.load_state_dict(torch.load(best_model_path, map_location=device))

# # Move model to device
# model.to(device)
# model.eval()import torchvision.models as models
import torch.nn as nn
################## For ResNet-50###############
# Load ResNet-50 instead of DenseNet-121
model = models.resnet50(pretrained=False)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, len(disease_names))

# Load weights
model.load_state_dict(torch.load(best_model_path, map_location=device))

# Move model to device and set to eval mode
model.to(device)
model.eval()

print(f"Loaded model from {best_model_path}")

train_df_full = pd.read_csv("/mnt/Internal/MedImage/chexpert_balanced_100_per_label_dis+demog.csv")
testing_df = train_df_full
# Define subgroups
gender_groups = {"Male": testing_df[testing_df['Sex'] == 'Male'],
                 "Female": testing_df[testing_df['Sex'] == 'Female']}

age_bins = [0, 30, 50, 70, np.inf]
age_labels = ['0-30', '31-50', '51-70', '71+']
testing_df['Age_Group'] = pd.cut(testing_df['Age'], bins=age_bins, labels=age_labels)
age_groups = {label: testing_df[testing_df['Age_Group'] == label] for label in age_labels}

race_groups = {race: testing_df[testing_df[race] == 1] for race in testing_df.columns if "PRIMARY_RACE_" in race}

# Merge all subgroups
subgroups = {**gender_groups, **age_groups, **race_groups}

# Define the image root directory (update this to your dataset location)
image_root = "/mnt/Internal/MedImage/CheXpert Dataset/unzip_chexpert_images/"  # <-- Change this to the actual path

class CustomDataset(Dataset):
    def __init__(self, df, image_root):
        self.df = df
        self.image_root = image_root  # Store the image root path
        self.image_paths = df['Path'].values  # Ensure 'Path' contains relative paths
        self.labels = df[disease_names].values.astype(np.float32)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        image_path = os.path.join(self.image_root, self.image_paths[idx])  # Combine root with relative path
        image = self.load_image(image_path)
        label = torch.tensor(self.labels[idx], dtype=torch.float32)
        return image, label

    def load_image(self, path):
        transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor()
        ])
        return transform(Image.open(path).convert('RGB'))

# Example Usage
dataset = CustomDataset(testing_df, image_root)


# Function to compute AUC
def compute_auc(model, dataset, device):
    dataloader = DataLoader(dataset, batch_size=2, shuffle=False)
    y_true, y_pred = [], []
    
    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            outputs = torch.sigmoid(model(images))  # Apply sigmoid activation
            y_true.append(labels.cpu().numpy())
            y_pred.append(outputs.cpu().numpy())
    
    y_true = np.vstack(y_true)
    y_pred = np.vstack(y_pred)
    auc_scores = {disease: roc_auc_score(y_true[:, i], y_pred[:, i]) for i, disease in enumerate(disease_names)}
    return auc_scores

# Compute AUC for each subgroup
auc_results = {}
for group, df in subgroups.items():
    if not df.empty:
        print(f"Processing subgroup: {group}")
        dataset = CustomDataset(df, image_root)  # Ensure you pass image_root
        auc_results[group] = compute_auc(model, dataset, device)

# Convert results to DataFrame and print
auc_df = pd.DataFrame(auc_results).T
print("Subgroup AUC analysis complete. Here are the results:")
print(auc_df)

Loaded model from /mnt/Internal/MedImage/CheXpert Dataset/Lab_Rotation_2/resnet50_Real_vs_real_training_for_14_disease/model_epoch_3.pth
Processing subgroup: Male
Processing subgroup: Female
Processing subgroup: 0-30
Processing subgroup: 31-50
Processing subgroup: 51-70
Processing subgroup: 71+
Processing subgroup: PRIMARY_RACE_American Indian or Alaska Native
Processing subgroup: PRIMARY_RACE_Asian
Processing subgroup: PRIMARY_RACE_Asian - Historical Conv


/home/dawood/lab2_rotaion/.conda/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/dawood/lab2_rotaion/.conda/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/dawood/lab2_rotaion/.conda/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/dawood/lab2_rotaion/.conda/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


Processing subgroup: PRIMARY_RACE_Asian, Hispanic


/home/dawood/lab2_rotaion/.conda/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/dawood/lab2_rotaion/.conda/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/dawood/lab2_rotaion/.conda/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


Processing subgroup: PRIMARY_RACE_Asian, non-Hispanic
Processing subgroup: PRIMARY_RACE_Black or African American
Processing subgroup: PRIMARY_RACE_Black, Hispanic


...

Processing subgroup: PRIMARY_RACE_Native Hawaiian or Other Pacific Islander
Processing subgroup: PRIMARY_RACE_Other
Processing subgroup: PRIMARY_RACE_Other, Hispanic
Processing subgroup: PRIMARY_RACE_Other, non-Hispanic
Processing subgroup: PRIMARY_RACE_Pacific Islander, Hispanic


/home/dawood/lab2_rotaion/.conda/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/dawood/lab2_rotaion/.conda/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/dawood/lab2_rotaion/.conda/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/dawood/lab2_rotaion/.conda/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/dawood/lab2_rotaion/.conda/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is

Processing subgroup: PRIMARY_RACE_Pacific Islander, non-Hispanic
Processing subgroup: PRIMARY_RACE_Patient Refused
Processing subgroup: PRIMARY_RACE_Race and Ethnicity Unknown
Processing subgroup: PRIMARY_RACE_Unknown
Processing subgroup: PRIMARY_RACE_White
Processing subgroup: PRIMARY_RACE_White or Caucasian


/home/dawood/lab2_rotaion/.conda/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/dawood/lab2_rotaion/.conda/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/dawood/lab2_rotaion/.conda/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


Processing subgroup: PRIMARY_RACE_White, Hispanic
Processing subgroup: PRIMARY_RACE_White, non-Hispanic
Subgroup AUC analysis complete. Here are the results:
                                                    No Finding  \
Male                                                  0.733838   
Female                                                0.717237   
0-30                                                  0.714470   
31-50                                                 0.706996   
51-70                                                 0.729440   
71+                                                   0.665599   
PRIMARY_RACE_American Indian or Alaska Native         0.773479   
PRIMARY_RACE_Asian                                    0.743805   
PRIMARY_RACE_Asian - Historical Conv                       NaN   
PRIMARY_RACE_Asian, Hispanic                          0.919355   
PRIMARY_RACE_Asian, non-Hispanic                      0.724393   
PRIMARY_RACE_Black or African American            

In [35]:
import pandas as pd

# 🔸 Paste your raw text block here
raw_text = """
Male                                                       0.701317  
Female                                                     0.715329  
0-30                                                       0.674673  
31-50                                                      0.684020  
51-70                                                      0.710833  
71+                                                        0.716656  
PRIMARY_RACE_American Indian or Alaska Native              0.712963  
PRIMARY_RACE_Asian                                         0.729705  
PRIMARY_RACE_Asian - Historical Conv                       0.666667  
PRIMARY_RACE_Asian, Hispanic                               0.860000  
PRIMARY_RACE_Asian, non-Hispanic                           0.766267  
PRIMARY_RACE_Black or African American                     0.694620  
PRIMARY_RACE_Black, Hispanic                               0.711538  
PRIMARY_RACE_Black, non-Hispanic                           0.731052  
PRIMARY_RACE_Native American, Hispanic                     0.384615  
PRIMARY_RACE_Native American, non-Hispanic                 0.730000  
PRIMARY_RACE_Native Hawaiian or Other Pacific I...         0.676220  
PRIMARY_RACE_Other                                         0.691636  
PRIMARY_RACE_Other, Hispanic                               0.733484  
PRIMARY_RACE_Other, non-Hispanic                           0.658577  
PRIMARY_RACE_Pacific Islander, Hispanic                    0.750000  
PRIMARY_RACE_Pacific Islander, non-Hispanic                0.668240  
PRIMARY_RACE_Patient Refused                               0.649638  
PRIMARY_RACE_Race and Ethnicity Unknown                    0.713454  
PRIMARY_RACE_Unknown                                       0.697882  
PRIMARY_RACE_White                                         0.692462  
PRIMARY_RACE_White or Caucasian                            0.357143  
PRIMARY_RACE_White, Hispanic                               0.700443  
PRIMARY_RACE_White, non-Hispanic                           0.708085 
"""

# 🔹 Parse manually line-by-line
lines = raw_text.strip().split('\n')
data = {}
for line in lines:
    parts = line.rsplit(None, 1)  # split only on the last space
    if len(parts) == 2:
        key, val = parts
        try:
            data[key.strip()] = float(val.strip())  # Attempt to convert to float
        except ValueError:  # If conversion fails, skip this line
            continue

# 🔹 Convert to DataFrame
df = pd.DataFrame(list(data.items()), columns=['Subgroup', 'Value'])

# 🔹 Group mapping
def map_to_group(subgroup):
    subgroup = subgroup.lower()
    if 'asian' in subgroup:
        return 'Asian'
    elif 'black' in subgroup:
        return 'Black'
    elif 'white' in subgroup:
        return 'White'
    elif subgroup.strip() == 'male':
        return 'Male'
    elif subgroup.strip() == 'female':
        return 'Female'
    elif '0-30' in subgroup:
        return '0-30'
    elif '31-50' in subgroup:
        return '31-50'
    elif '51-70' in subgroup:
        return '51-70'
    elif '71+' in subgroup or '71' in subgroup:
        return '71+'
    else:
        return None

df['Group'] = df['Subgroup'].apply(map_to_group)

# 🔹 Drop unmatched
df_filtered = df.dropna(subset=['Group'])

# 🔹 Group and average
grouped = df_filtered.groupby('Group')['Value'].mean()

# 🔹 Final formatting
ordered = ['Male', 'Female', '0-30', '31-50', '51-70', '71+', 'Asian', 'Black', 'White']
final = grouped.reindex(ordered)
final_df = pd.DataFrame([final.values], columns=ordered)
final_df.insert(0, 'Disease', 'SD')

# 🔹 Save
final_df.to_csv("SD.csv", index=False)

# 🔹 Show output
print("\n✅ Done! Final DataFrame:")
print(final_df)



✅ Done! Final DataFrame:
  Disease      Male    Female      0-30    31-50     51-70       71+  \
0      SD  0.701317  0.715329  0.674673  0.68402  0.710833  0.716656   

      Asian     Black    White  
0  0.675956  0.712403  0.70033  
